# US Rates & SOFR — Interactive Live Analysis

**Bhavesh Anchalia**

Adjust the sliders and press **Run Interact** to explore model sensitivities in real time.  
Each section is self-contained — no FRED key required.

| Section | Widget type |
|---------|------------|
| 1. SOFR Curve Builder | Sliders (ON rate + futures cuts) |
| 2. Convexity Adjustment | Sliders (σ, mean reversion) |
| 3. Taylor Rule Gap | Sliders (macro variables) |
| 4. Nelson-Siegel Fitting | Yield inputs → NS fit |
| 5. Composite Signal | Macro inputs → trading signal |
| 6. FOMC Probabilities | Rate inputs → cut/hold/hike probs |

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import date
from dateutil.relativedelta import relativedelta
import ipywidgets as widgets
from IPython.display import display, clear_output

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (11, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
print('✓ Setup complete — all widgets ready')

## 1. SOFR Curve Builder

Bootstrap a SOFR OIS forward curve from the overnight fixing and an implied futures strip.  
The three cut sliders represent the cumulative easing priced into the first three quarterly contracts.

In [ ]:
from sofr_engine import SOFRCurveBootstrapper

def _build_futures(sofr_on_pct, cut1_bps, cut2_bps, cut3_bps):
    """Build a synthetic 8-contract SR3 futures strip from cut assumptions."""
    r = sofr_on_pct / 100
    cum1 = cut1_bps / 10000
    cum2 = (cut1_bps + cut2_bps) / 10000
    cum3 = (cut1_bps + cut2_bps + cut3_bps) / 10000
    today = date.today()
    # Approximate IMM quarterly dates
    rows = []
    for i in range(8):
        exp  = today + relativedelta(months=3*(i+1))
        acc  = today + relativedelta(months=3*(i+2))
        if i < 2:
            impl = r - cum1
        elif i < 4:
            impl = r - cum2
        else:
            impl = r - cum3
        rows.append({'expiry': exp, 'accrual_end': acc, 'implied_rate': max(impl, 0.001)})
    return pd.DataFrame(rows)

out_curve = widgets.Output()

@widgets.interact_manual(
    sofr_on  = widgets.FloatSlider(value=4.33, min=1.0, max=6.5, step=0.05,
                                    description='ON SOFR %', style={'description_width': '120px'}, layout=widgets.Layout(width='450px')),
    cut1_bps = widgets.IntSlider(value=25, min=0, max=75, step=5,
                                  description='Cut 1 (bps)', style={'description_width': '120px'}, layout=widgets.Layout(width='450px')),
    cut2_bps = widgets.IntSlider(value=25, min=0, max=75, step=5,
                                  description='Cut 2 (bps)', style={'description_width': '120px'}, layout=widgets.Layout(width='450px')),
    cut3_bps = widgets.IntSlider(value=25, min=0, max=75, step=5,
                                  description='Cut 3 (bps)', style={'description_width': '120px'}, layout=widgets.Layout(width='450px')),
)
def plot_sofr_curve(sofr_on=4.33, cut1_bps=25, cut2_bps=25, cut3_bps=25):
    with out_curve:
        clear_output(wait=True)
        try:
            r = sofr_on / 100
            cum3 = (cut1_bps + cut2_bps + cut3_bps) / 10000
            futures = _build_futures(sofr_on, cut1_bps, cut2_bps, cut3_bps)
            # OIS quotes: longer end prices in the full cut path + small term premium
            long_rate = r - cum3
            ois_quotes = [
                (2.0,  long_rate + 0.002),
                (3.0,  long_rate + 0.001),
                (5.0,  long_rate - 0.001),
                (7.0,  long_rate - 0.002),
                (10.0, long_rate - 0.003),
                (15.0, long_rate - 0.003),
                (20.0, long_rate - 0.003),
                (30.0, long_rate - 0.004),
            ]
            curve = SOFRCurveBootstrapper.from_market_data(
                date.today(), r, futures, ois_quotes, sigma=0.010
            )
            tenors = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30])
            zeros  = np.array([curve.zero_rate(t) * 100 for t in tenors])
            fwds   = np.array([curve.forward_rate(t, t + 0.25) * 100 for t in tenors])

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
            ax1.plot(tenors, zeros, 'b-o', markersize=5, label='Zero rate')
            ax1.plot(tenors, fwds,  'r--s', markersize=5, label='3M forward rate')
            ax1.axhline(sofr_on, color='grey', linestyle=':', alpha=0.6, label=f'ON SOFR {sofr_on:.2f}%')
            ax1.set_xlabel('Tenor (years)')
            ax1.set_ylabel('Rate (%)')
            ax1.set_title('SOFR Forward Curve')
            ax1.legend(fontsize=9)

            table_data = {'Tenor': [f'{t}Y' for t in tenors],
                          'Zero (%)': [f'{z:.3f}' for z in zeros],
                          '3M Fwd (%)': [f'{f:.3f}' for f in fwds]}
            ax2.axis('off')
            tbl = ax2.table(cellText=list(zip(table_data['Tenor'],
                                              table_data['Zero (%)'],
                                              table_data['3M Fwd (%)'])),
                             colLabels=['Tenor', 'Zero (%)', '3M Fwd (%)'],
                             loc='center', cellLoc='center')
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(10)
            tbl.scale(1.2, 1.4)
            ax2.set_title(f'Curve as of {date.today()}  |  Total cuts priced: {cut1_bps+cut2_bps+cut3_bps}bps',
                          fontsize=10)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Error building curve: {e}')

display(out_curve)

## 2. Hull-White Convexity Adjustment

For CME SR3 futures: the futures rate exceeds the forward rate by a convexity adjustment  
CA = ½σ²B(0,T₁)B(0,T₂) where B(0,T) = (1−e^{−aT})/a.

Higher volatility or shorter mean reversion both increase the adjustment.

In [ ]:
from sofr_engine.convexity import hull_white_convexity_adjustment

out_ca = widgets.Output()

@widgets.interact(
    sigma_pct   = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                       description='σ (%)', style={'description_width': '120px'},
                                       layout=widgets.Layout(width='450px')),
    mean_rev_pct= widgets.FloatSlider(value=5.0, min=0.5, max=30.0, step=0.5,
                                       description='a (%)', style={'description_width': '120px'},
                                       layout=widgets.Layout(width='450px')),
)
def plot_convexity(sigma_pct=1.0, mean_rev_pct=5.0):
    with out_ca:
        clear_output(wait=True)
        sigma = sigma_pct / 100
        a     = mean_rev_pct / 100
        contracts = [(f'Mar-{25+i//4}', 0.25*(i+1), 0.25*(i+2)) for i in range(8)]
        labels = [c[0] for c in contracts]
        cas    = [hull_white_convexity_adjustment(c[1], c[2], sigma=sigma, mean_reversion=a) * 10000
                  for c in contracts]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
        bars = ax1.bar(labels, cas, color='steelblue', alpha=0.8, edgecolor='navy', linewidth=0.5)
        for bar, val in zip(bars, cas):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        ax1.set_ylabel('Convexity Adjustment (bps)')
        ax1.set_title(f'Hull-White CA  |  σ={sigma_pct:.1f}%  a={mean_rev_pct:.1f}%')
        ax1.tick_params(axis='x', rotation=30)

        # Sensitivity: CA vs sigma at fixed a
        sig_range = np.linspace(0.1, 3.0, 60) / 100
        ca_last = [hull_white_convexity_adjustment(1.75, 2.0, s, a) * 10000 for s in sig_range]
        ax2.plot(sig_range * 100, ca_last, 'b-', linewidth=2, label='Sep contract (T1=1.75Y)')
        ax2.axvline(sigma_pct, color='red', linestyle='--', alpha=0.7, label=f'Current σ={sigma_pct:.1f}%')
        ax2.set_xlabel('σ (%)')
        ax2.set_ylabel('CA (bps)')
        ax2.set_title('CA Sensitivity to Volatility')
        ax2.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

        print(f'\nNote: As a→0 (σ={sigma_pct:.1f}%): CA → ½σ²T₁T₂ = '
              f'{0.5*(sigma**2)*1.75*2.0*10000:.3f}bps  '
              f'(current: {hull_white_convexity_adjustment(1.75, 2.0, sigma, a)*10000:.3f}bps)')

display(out_ca)

## 3. Taylor Rule — Policy Gap Explorer

The standard Taylor Rule: **r* = r_neutral + 0.5(π − π*) + 0.5(−2(u − u*))**

The policy gap = EFFR − r*. A large positive gap signals the Fed is restrictive → bullish duration.

In [ ]:
from models.taylor_rule import compute_taylor_rule, TaylorRuleConfig

out_taylor = widgets.Output()

@widgets.interact_manual(
    inflation    = widgets.FloatSlider(value=2.11, min=0.0, max=8.0, step=0.1,
                                        description='Core PCE %', style={'description_width': '130px'},
                                        layout=widgets.Layout(width='460px')),
    unemployment = widgets.FloatSlider(value=4.30, min=2.0, max=8.0, step=0.1,
                                        description='Unemployment %', style={'description_width': '130px'},
                                        layout=widgets.Layout(width='460px')),
    neutral_rate = widgets.FloatSlider(value=2.50, min=0.5, max=5.0, step=0.25,
                                        description='Neutral r* %', style={'description_width': '130px'},
                                        layout=widgets.Layout(width='460px')),
    effr         = widgets.FloatSlider(value=4.33, min=0.0, max=7.0, step=0.05,
                                        description='EFFR %', style={'description_width': '130px'},
                                        layout=widgets.Layout(width='460px')),
    variant      = widgets.Dropdown(options=['standard', 'balanced'], value='standard',
                                     description='Variant', style={'description_width': '130px'}),
)
def plot_taylor(inflation=2.11, unemployment=4.30, neutral_rate=2.50, effr=4.33, variant='standard'):
    with out_taylor:
        clear_output(wait=True)
        cfg = TaylorRuleConfig(neutral_rate=neutral_rate, inflation_target=2.0,
                               nairu=4.0, variant=variant)
        idx  = pd.date_range('2025-01-01', periods=1, freq='MS')
        infl = pd.Series([inflation],    index=idx)
        unem = pd.Series([unemployment], index=idx)
        res  = compute_taylor_rule(infl, unem, cfg)
        taylor_rate = res['taylor_rate'].iloc[0]
        infl_gap    = res['inflation_gap'].iloc[0]
        unem_gap    = res['unemployment_gap'].iloc[0]
        gap_bps     = (effr - taylor_rate) * 100

        if gap_bps > 75:
            signal, color = 'LONG DURATION  (Fed restrictive)', '#1a9641'
        elif gap_bps < -75:
            signal, color = 'SHORT DURATION  (Fed accommodative)', '#d7191c'
        else:
            signal, color = 'NEUTRAL  (within tolerance band)', '#f4a11d'

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

        # Waterfall decomposition
        components = ['Neutral r*', 'Inflation\ngap contrib', 'Output\ngap contrib', 'Taylor\nRate', 'EFFR']
        infl_contrib = 0.5 * infl_gap
        outp_contrib = unem_gap  # already has output_gap weight baked in
        values       = [neutral_rate, infl_contrib, outp_contrib, None, effr]
        bar_values   = [neutral_rate, infl_contrib, outp_contrib, taylor_rate, effr]
        bar_colors   = ['#4393c3', '#92c5de' if infl_contrib >= 0 else '#f4a582',
                         '#92c5de' if outp_contrib >= 0 else '#f4a582',
                         '#2166ac', '#d6604d']
        bars = ax1.bar(components, bar_values, color=bar_colors, edgecolor='white', linewidth=0.5, alpha=0.9)
        ax1.axhline(taylor_rate, color='#2166ac', linestyle='--', linewidth=1.2, alpha=0.5)
        for bar, val in zip(bars, bar_values):
            ax1.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.04 if val >= 0 else bar.get_height() - 0.15,
                     f'{val:+.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax1.set_ylabel('Rate (%)')
        ax1.set_title(f'Taylor Rule Decomposition  ({variant})')
        ax1.set_ylim(min(0, min(bar_values) - 0.5), max(bar_values) + 0.7)

        # Gap sensitivity vs inflation
        infl_range = np.linspace(0.5, 6.0, 100)
        gaps = []
        for pi in infl_range:
            r = compute_taylor_rule(
                pd.Series([pi], index=idx), unem, cfg
            )['taylor_rate'].iloc[0]
            gaps.append((effr - r) * 100)
        ax2.plot(infl_range, gaps, 'b-', linewidth=2)
        ax2.axhline(75,  color='green',  linestyle='--', alpha=0.6, label='+75bp (long threshold)')
        ax2.axhline(-75, color='red',    linestyle='--', alpha=0.6, label='-75bp (short threshold)')
        ax2.axhline(0,   color='grey',   linestyle=':',  alpha=0.5)
        ax2.axvline(inflation, color='orange', linestyle='-', alpha=0.8,
                    label=f'Current π={inflation:.2f}%')
        ax2.scatter([inflation], [gap_bps], color='orange', s=80, zorder=5)
        ax2.set_xlabel('Core PCE (%)')
        ax2.set_ylabel('Policy Gap (bps)')
        ax2.set_title('Policy Gap vs Inflation  (u held constant)')
        ax2.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        print(f'  Taylor Rate:  {taylor_rate:.3f}%  |  EFFR: {effr:.2f}%  |  Gap: {gap_bps:+.1f}bps')
        print(f'  Signal: {signal}', flush=True)

display(out_taylor)

## 4. Nelson-Siegel Treasury Curve Fitting

Enter your own Treasury yield quotes (in %). The Nelson-Siegel model fits β₀ (level), β₁ (slope), β₂ (curvature), λ (decay).

Regime classification: β₁ < −1 → normal_steep; −1 ≤ β₁ < 0 → normal_flat; 0 ≤ β₁ < 1 → flat/transitional; β₁ ≥ 1 → inverted.

In [ ]:
from models.nelson_siegel import fit_nelson_siegel, ns_yield, NSParams, classify_curve_regime

out_ns = widgets.Output()

style  = {'description_width': '80px'}
layout = widgets.Layout(width='200px')

w_y1m  = widgets.FloatText(value=5.35, description='1M (%)',  style=style, layout=layout)
w_y3m  = widgets.FloatText(value=5.30, description='3M (%)',  style=style, layout=layout)
w_y6m  = widgets.FloatText(value=5.15, description='6M (%)',  style=style, layout=layout)
w_y1y  = widgets.FloatText(value=4.90, description='1Y (%)',  style=style, layout=layout)
w_y2y  = widgets.FloatText(value=4.50, description='2Y (%)',  style=style, layout=layout)
w_y5y  = widgets.FloatText(value=4.25, description='5Y (%)',  style=style, layout=layout)
w_y10y = widgets.FloatText(value=4.35, description='10Y (%)', style=style, layout=layout)
w_y20y = widgets.FloatText(value=4.55, description='20Y (%)', style=style, layout=layout)
w_y30y = widgets.FloatText(value=4.50, description='30Y (%)', style=style, layout=layout)

btn_ns = widgets.Button(description='Fit Nelson-Siegel', button_style='primary',
                         layout=widgets.Layout(width='200px', margin='10px 0'))

grid = widgets.GridBox(
    [w_y1m, w_y3m, w_y6m, w_y1y, w_y2y, w_y5y, w_y10y, w_y20y, w_y30y],
    layout=widgets.Layout(grid_template_columns='repeat(3, 210px)', grid_gap='4px')
)
display(widgets.VBox([widgets.HTML('<b>Enter Treasury yields (%):</b>'), grid, btn_ns]))

def on_fit_click(b):
    with out_ns:
        clear_output(wait=True)
        mats   = np.array([1/12, 3/12, 6/12, 1, 2, 5, 10, 20, 30])
        yields = np.array([w_y1m.value, w_y3m.value, w_y6m.value, w_y1y.value,
                           w_y2y.value, w_y5y.value, w_y10y.value, w_y20y.value, w_y30y.value])
        try:
            params = fit_nelson_siegel(mats, yields)
            t_fit  = np.linspace(0.08, 30, 300)
            y_fit  = ns_yield(t_fit, params)
            y_obs  = ns_yield(mats, params)
            rmse   = np.sqrt(np.mean((y_obs - yields)**2))

            ns_df  = pd.DataFrame({'beta1': [params.beta1], 'beta0': [params.beta0]})
            regime = classify_curve_regime(ns_df)[0]

            fig, ax = plt.subplots(figsize=(11, 4.5))
            ax.scatter(mats, yields, s=80, zorder=5, color='#d7191c', label='Observed CMT yields')
            ax.plot(t_fit, y_fit, 'b-', linewidth=2.5,
                    label=f'NS fit  β₀={params.beta0:.3f}  β₁={params.beta1:.3f}  β₂={params.beta2:.3f}  λ={params.lam:.3f}')
            ax.set_xlabel('Maturity (years)')
            ax.set_ylabel('Yield (%)')
            ax.set_title(f'Nelson-Siegel Fit  |  RMSE={rmse:.4f}%  |  Regime: {regime}')
            ax.legend(fontsize=9)
            plt.tight_layout()
            plt.show()

            print(f'\n  β₀ (level):     {params.beta0:>8.4f}%  — long-run yield')
            print(f'  β₁ (slope):     {params.beta1:>8.4f}%  — (−) = normal, (+) = inverted')
            print(f'  β₂ (curvature): {params.beta2:>8.4f}%  — hump shape')
            print(f'  λ  (decay):     {params.lam:>8.4f}   — hump location')
            print(f'  RMSE:           {rmse:>8.4f}%')
            print(f'  Regime:         {regime}')
        except Exception as e:
            print(f'Fit failed: {e}')

btn_ns.on_click(on_fit_click)
display(out_ns)

## 5. Composite Signal Monitor

Four sub-signals feed the composite:
1. **Policy gap** — Fed vs Taylor rate (EFFR − r*)
2. **Inflation momentum** — Core PCE trend vs 2% target
3. **Labor market** — Unemployment vs NAIRU
4. **Curve slope** — 2s10s spread

Position = −1 (short) / 0 (flat) / +1 (long). Long = hold duration (buy bonds).

In [ ]:
from models.macro_signals import (
    policy_gap_signal, inflation_momentum_signal,
    labor_market_signal, curve_slope_signal, composite_signal,
)

out_sig = widgets.Output()

@widgets.interact_manual(
    core_pce    = widgets.FloatSlider(value=2.11, min=0.5, max=7.0, step=0.1,
                                       description='Core PCE %', style={'description_width': '130px'},
                                       layout=widgets.Layout(width='460px')),
    unemployment= widgets.FloatSlider(value=4.30, min=2.0, max=8.0, step=0.1,
                                       description='Unemployment %', style={'description_width': '130px'},
                                       layout=widgets.Layout(width='460px')),
    tsy_2y      = widgets.FloatSlider(value=4.50, min=1.0, max=7.0, step=0.05,
                                       description='2Y Yield %', style={'description_width': '130px'},
                                       layout=widgets.Layout(width='460px')),
    tsy_10y     = widgets.FloatSlider(value=4.35, min=1.0, max=7.0, step=0.05,
                                       description='10Y Yield %', style={'description_width': '130px'},
                                       layout=widgets.Layout(width='460px')),
    effr        = widgets.FloatSlider(value=4.33, min=0.0, max=7.0, step=0.05,
                                       description='EFFR %', style={'description_width': '130px'},
                                       layout=widgets.Layout(width='460px')),
    taylor_rate = widgets.FloatSlider(value=2.26, min=0.0, max=6.0, step=0.05,
                                       description='Taylor Rate %', style={'description_width': '130px'},
                                       layout=widgets.Layout(width='460px')),
)
def plot_signals(core_pce=2.11, unemployment=4.30, tsy_2y=4.50,
                 tsy_10y=4.35, effr=4.33, taylor_rate=2.26):
    with out_sig:
        clear_output(wait=True)
        n   = 252
        idx = pd.date_range('2024-01-01', periods=n, freq='B')

        def s(v): return pd.Series([v] * n, index=idx)

        sig_pg   = policy_gap_signal(s(effr), s(taylor_rate))
        sig_infl = inflation_momentum_signal(s(core_pce))
        sig_lab  = labor_market_signal(s(unemployment))
        slope    = tsy_10y - tsy_2y
        sig_slp  = curve_slope_signal(s(slope))

        df = pd.DataFrame({
            'core_pce_yoy': core_pce, 'unemployment': unemployment,
            'tsy_2y': tsy_2y, 'tsy_10y': tsy_10y,
            'taylor_rate': taylor_rate, 'effr': effr,
        }, index=idx)
        comp = composite_signal(df)
        pos  = comp['position'].iloc[-1]

        # Display
        signals = {
            'Policy Gap\n(EFFR vs Taylor)': sig_pg.iloc[-1],
            'Inflation\nMomentum': sig_infl.iloc[-1],
            'Labor\nMarket': sig_lab.iloc[-1],
            'Curve Slope\n(2s10s)': sig_slp.iloc[-1],
        }
        color_map = {1: '#1a9641', 0: '#f4a11d', -1: '#d7191c'}
        label_map = {1: 'LONG (+1)', 0: 'FLAT (0)', -1: 'SHORT (−1)'}

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

        names  = list(signals.keys())
        vals   = list(signals.values())
        colors = [color_map[v] for v in vals]
        ax1.barh(names, vals, color=colors, edgecolor='white', linewidth=0.5)
        ax1.set_xlim(-1.5, 1.5)
        ax1.axvline(0, color='black', linewidth=0.8)
        ax1.set_xlabel('Signal (−1 = Short, 0 = Flat, +1 = Long)')
        ax1.set_title('Individual Sub-Signals')
        for i, (v, nm) in enumerate(zip(vals, names)):
            ax1.text(v + (0.05 if v >= 0 else -0.05), i,
                     label_map[int(v)], va='center', ha='left' if v >= 0 else 'right', fontsize=9)

        # Composite gauge
        ax2.axis('off')
        pos_color = color_map[int(pos)]
        pos_label = label_map[int(pos)]
        ax2.text(0.5, 0.65, 'COMPOSITE POSITION', ha='center', va='center',
                  fontsize=13, transform=ax2.transAxes, color='#333333')
        ax2.text(0.5, 0.42, pos_label, ha='center', va='center',
                  fontsize=32, fontweight='bold', transform=ax2.transAxes, color=pos_color)
        agreement = abs(sum(vals)) / len(vals)
        ax2.text(0.5, 0.22, f'Agreement: {agreement:.0%}  |  2s10s slope: {slope*100:.0f}bps',
                  ha='center', va='center', fontsize=11, transform=ax2.transAxes, color='#555555')
        ax2.text(0.5, 0.10, f'Gap vs Taylor: {(effr - taylor_rate)*100:+.0f}bps  |  PCE: {core_pce:.2f}%  |  U: {unemployment:.2f}%',
                  ha='center', va='center', fontsize=9, transform=ax2.transAxes, color='#777777')
        plt.tight_layout()
        plt.show()

display(out_sig)

## 6. FOMC Outcome Probabilities

Fed Funds futures and OIS pricing imply a probability distribution over FOMC outcomes.  
The implied rate is read from the front-month SR1 futures contract or 30-day Fed Funds futures.

In [ ]:
from models.fomc_probability import fedwatch_probabilities, fomc_prob_summary

out_fomc = widgets.Output()

@widgets.interact(
    current_rate = widgets.FloatSlider(value=4.33, min=0.0, max=7.0, step=0.05,
                                        description='Current FFR %', style={'description_width': '140px'},
                                        layout=widgets.Layout(width='480px')),
    implied_rate = widgets.FloatSlider(value=4.08, min=0.0, max=7.0, step=0.05,
                                        description='SR1-Implied %', style={'description_width': '140px'},
                                        layout=widgets.Layout(width='480px')),
)
def plot_fomc(current_rate=4.33, implied_rate=4.08):
    with out_fomc:
        clear_output(wait=True)
        try:
            probs   = fedwatch_probabilities(implied_rate/100, current_rate/100)
            summary = fomc_prob_summary(implied_rate/100, current_rate/100)
            outcomes = sorted(probs.keys())
            prob_vals = [probs[o] for o in outcomes]
            labels    = [f'{o:+d}bp' for o in outcomes]
            colors    = ['#1a9641' if o < 0 else ('#d7191c' if o > 0 else '#f4a11d') for o in outcomes]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
            bars = ax1.barh(labels, prob_vals, color=colors, edgecolor='white', linewidth=0.5)
            for bar, val in zip(bars, prob_vals):
                ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                         f'{val:.1%}', va='center', fontsize=10)
            ax1.set_xlim(0, 1.15)
            ax1.set_xlabel('Probability')
            ax1.set_title(f'FOMC Outcome Distribution\nCurrent: {current_rate:.2f}%  |  Implied: {implied_rate:.2f}%')
            ax1.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

            # Summary panel
            ax2.axis('off')
            move_bps = (implied_rate - current_rate) * 100
            ax2.text(0.5, 0.88, 'Market Pricing Summary', ha='center', fontsize=13,
                      transform=ax2.transAxes, fontweight='bold')
            rows = [
                ('Prob(Cut)',  f"{summary['p_cut']:.1%}",  '#1a9641'),
                ('Prob(Hold)', f"{summary['p_hold']:.1%}", '#f4a11d'),
                ('Prob(Hike)', f"{summary['p_hike']:.1%}", '#d7191c'),
                ('Implied move', f'{move_bps:+.1f}bps', '#333333'),
            ]
            for i, (lbl, val, col) in enumerate(rows):
                y = 0.68 - i * 0.16
                ax2.text(0.25, y, lbl + ':', ha='right', fontsize=12, transform=ax2.transAxes, color='#555')
                ax2.text(0.30, y, val, ha='left',  fontsize=13, fontweight='bold',
                          transform=ax2.transAxes, color=col)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Error: {e}')

display(out_fomc)